In [26]:
import os
import pandas as pd

# Path ke folder yang berisi file-file DataFrame
folder_path = '../comodity-price-prediction-penyisihan-arkavidia-9/Global Commodity Price'

# Dictionary untuk menyimpan semua DataFrame dengan nama file sebagai key
df_dict = {}

# Loop melalui semua file dalam folder
for file_name in os.listdir(folder_path):
    # Pastikan file yang dibaca adalah file yang diinginkan (misalnya, CSV)
    if file_name.endswith('.csv'):
        # Buat path lengkap ke file
        file_path = os.path.join(folder_path, file_name)
        
        # Baca file dan simpan ke dictionary dengan nama file sebagai key
        df = pd.read_csv(file_path)
        df_dict[file_name] = df  # Gunakan nama file sebagai key


In [27]:
df_crude_oil_wti =df_dict['Crude Oil WTI Futures Historical Data.csv']
df_natural_gas = df_dict['Natural Gas Futures Historical Data.csv']
df_newcastle_coal = df_dict['Newcastle Coal Futures Historical Data.csv']
df_palm_oil = df_dict['Palm Oil Futures Historical Data.csv']
df_US_sugar_11 = df_dict['US Sugar 11 Futures Historical Data.csv']
df_US_Wheat = df_dict['US Wheat Futures Historical Data.csv']

In [28]:
df_dict_sorted = {}

for key, df in df_dict.items():
    df_sorted = df.sort_values(by='Date', ascending=True).reset_index(drop=True)
    
    # Pastikan kolom 'Date' dalam format datetime
    df_sorted['Date'] = pd.to_datetime(df_sorted['Date'])
    
    # Buat range tanggal lengkap dari 2022-01-01 sampai 2024-09-30
    full_date_range = pd.date_range(start='2022-01-01', end='2024-09-30', freq='D')
    
    # Set index ke 'Date' untuk interpolasi
    df_sorted = df_sorted.set_index('Date')
    
    # Konversi 'Change %' ke numerik (hapus % dan ubah ke float)
    if 'Change %' in df_sorted.columns:
        df_sorted['Change %'] = (
            df_sorted['Change %']
            .replace('%', '', regex=True)  # Hapus simbol %
            .astype(float)                 # Konversi ke float
        )
    
    # Reindex dataframe dan interpolasi nilai yang hilang
    df_sorted = df_sorted.reindex(full_date_range).interpolate()
    
    # Reset index agar kembali ke format semula
    df_sorted = df_sorted.reset_index().rename(columns={'index': 'Date'})
    
    # Simpan hasil ke dictionary
    df_dict_sorted[key] = df_sorted
    
    # Cek apakah masih ada missing values
    print(f"Missing values in {key}:\n", df_sorted.isnull().sum())

    # Tampilkan beberapa baris hasil setelah interpolasi
    print(df_sorted.head())


Missing values in Newcastle Coal Futures Historical Data.csv:
 Date          0
Price         2
Open          2
High          2
Low           2
Vol.        609
Change %      2
dtype: int64
        Date  Price   Open   High    Low   Vol.  Change %
0 2022-01-01    NaN    NaN    NaN    NaN    NaN       NaN
1 2022-01-02    NaN    NaN    NaN    NaN    NaN       NaN
2 2022-01-03  157.5  157.5  157.5  157.5  0.00K     -7.13
3 2022-01-04  174.1  172.0  174.0  171.0  0.27K     10.54
4 2022-01-05  179.9  180.0  180.0  180.0  0.02K      3.33
Missing values in US Sugar 11 Futures Historical Data.csv:
 Date          0
Price         2
Open          2
High          2
Low           2
Vol.        315
Change %      2
dtype: int64
        Date  Price   Open   High    Low    Vol.  Change %
0 2022-01-01    NaN    NaN    NaN    NaN     NaN       NaN
1 2022-01-02    NaN    NaN    NaN    NaN     NaN       NaN
2 2022-01-03  18.74  18.94  19.01  18.68  20.40K     -0.74
3 2022-01-04  18.75  18.75  18.85  18.62  4

/tmp/ipykernel_188279/3441311735.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_sorted = df_sorted.reindex(full_date_range).interpolate()
/tmp/ipykernel_188279/3441311735.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_sorted = df_sorted.reindex(full_date_range).interpolate()
/tmp/ipykernel_188279/3441311735.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_sorted = df_sorted.reindex(full_date_range).interpolate()
/tmp/ipykernel_188279/3441311735.py:24: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) b

In [29]:
import pandas as pd
import numpy as np

# Fungsi untuk konversi 'K' ke ribuan, 'M' ke jutaan
def convert_volume(value):
    if pd.isna(value):  # Tangani NaN
        return np.nan
    value = str(value).replace(',', '')  # Pastikan tidak ada koma
    if value.endswith('K'):
        return float(value[:-1]) * 1e3
    elif value.endswith('M'):
        return float(value[:-1]) * 1e6
    else:
        return float(value)  # Jika sudah dalam bentuk angka langsung return

In [30]:
for key, df_sorted in df_dict_sorted.items():
    df_sorted['Vol.'] = df_sorted['Vol.'].astype(str).replace('nan', 'NaN')
    df_sorted['Vol.'] = df_sorted['Vol.'].apply(convert_volume)
    df_sorted['Vol.'] = df_sorted['Vol.'].interpolate()  # Interpolasi nilai yang null
    df_dict_sorted[key] = df_sorted  # Simpan kembali ke dictionary


In [31]:
for key, df_sorted in df_dict_sorted.items():
    
    # Set index ke 'Date' untuk interpolasi
    df_sorted = df_sorted.set_index('Date')
    
    # Reindex dataframe dan isi nilai null dengan backward fill (bfill)
    df_sorted = df_sorted.reindex(full_date_range).bfill()
    
    # Reset index agar kembali ke format semula
    df_sorted = df_sorted.reset_index().rename(columns={'index': 'Date'})
    
    # Simpan hasil ke dictionary
    df_dict_sorted[key] = df_sorted


In [38]:
df_dict_sorted['Crude Oil WTI Futures Historical Data.csv']

,Date,Price,Open,High,Low,Vol.,Change %
0,2022-01-01,75.850000,75.530000,76.180000,74.010000,112230.000000,1.300000
1,2022-01-02,75.850000,75.530000,76.180000,74.010000,112230.000000,1.300000
2,2022-01-03,75.850000,75.530000,76.180000,74.010000,112230.000000,1.300000
3,2022-01-04,76.740000,75.800000,77.400000,75.470000,156450.000000,1.170000
4,2022-01-05,77.470000,76.880000,78.160000,76.250000,197970.000000,0.950000
...,...,...,...,...,...,...,...
999,2024-09-26,67.190000,69.300000,69.470000,66.490000,274690.000000,-2.810000
1000,2024-09-27,67.670000,66.950000,68.130000,66.580000,169900.000000,0.710000
1001,2024-09-28,67.703333,67.326667,68.363333,66.766667,176363.333333,0.523333
1002,2024-09-29,67.736667,67.703333,68.596667,66.953333,182826.666667,0.336667


In [33]:
df_US_Wheat

,Date,Price,Open,High,Low,Vol.,Change %
0,2024-09-30,584.00,579.75,590.75,575.50,58.87K,0.69%
1,2024-09-27,580.00,583.00,583.50,575.50,46.19K,-0.73%
2,2024-09-26,584.25,590.50,596.25,582.75,69.06K,-0.85%
3,2024-09-25,589.25,579.00,591.25,573.75,52.96K,1.95%
4,2024-09-24,578.00,581.25,589.25,575.25,52.27K,-0.77%
...,...,...,...,...,...,...,...
694,2022-01-07,758.50,748.00,760.75,735.50,58.46K,1.68%
695,2022-01-06,746.00,761.75,762.00,736.00,64.43K,-1.94%
696,2022-01-05,760.75,771.00,771.50,756.00,40.24K,-1.20%
697,2022-01-04,770.00,758.00,771.50,756.25,43.22K,1.58%


In [34]:
import pandas as pd

# Pastikan kolom 'Date' dalam format datetime
df_exchange['Date'] = pd.to_datetime(df_exchange['Date'])

# Buat range tanggal lengkap dari 2022-01-03 sampai 2024-09-30
full_date_range = pd.date_range(start='2022-01-01', end='2024-09-30', freq='D')

# Set index ke 'Date' untuk memastikan interpolasi berjalan dengan baik
df_exchange = df_exchange.set_index('Date')

# Reindex dataframe agar semua tanggal ada, lalu interpolasi nilai yang hilang
df_exchange_inter = df_exchange.reindex(full_date_range).interpolate()

# Reset index agar kembali ke format semula
df_exchange_inter = df_exchange_inter.reset_index().rename(columns={'index': 'Date'})

# Cek apakah masih ada missing values
print(df_exchange_inter.isnull().sum())


NameError: name 'df_exchange' is not defined

In [22]:
data_frames['df_US_Wheat']

,Date,Price,Open,High,Low,Vol.,Change %
0,2024-09-30,584.00,579.75,590.75,575.50,58.87K,0.69%
1,2024-09-27,580.00,583.00,583.50,575.50,46.19K,-0.73%
2,2024-09-26,584.25,590.50,596.25,582.75,69.06K,-0.85%
3,2024-09-25,589.25,579.00,591.25,573.75,52.96K,1.95%
4,2024-09-24,578.00,581.25,589.25,575.25,52.27K,-0.77%
...,...,...,...,...,...,...,...
694,2022-01-07,758.50,748.00,760.75,735.50,58.46K,1.68%
695,2022-01-06,746.00,761.75,762.00,736.00,64.43K,-1.94%
696,2022-01-05,760.75,771.00,771.50,756.00,40.24K,-1.20%
697,2022-01-04,770.00,758.00,771.50,756.25,43.22K,1.58%


In [23]:
import pandas as pd

# Baca file submission kamu dan sample submission
sample_submission = pd.read_csv('../comodity-price-prediction-penyisihan-arkavidia-9/Harga Bahan Pangan/train/Bawang Merah.csv')
sample_submission['Date'] = pd.to_datetime(sample_submission['Date'])
df_crude_oil_wti['Date'] = pd.to_datetime(df_crude_oil_wti['Date'])

# Bandingkan id yang ada di sample submission tapi tidak ada di submission
missing_ids = set(sample_submission['Date']) - set(df_crude_oil_wti['Date'])

# Tampilkan jumlah dan contoh id yang hilang
print(f"Jumlah id yang hilang: {len(missing_ids)}")
print("Contoh id yang hilang:", list(missing_ids)[:10])

Jumlah id yang hilang: 279
Contoh id yang hilang: [Timestamp('2022-12-31 00:00:00'), Timestamp('2022-01-30 00:00:00'), Timestamp('2024-02-17 00:00:00'), Timestamp('2024-04-21 00:00:00'), Timestamp('2024-02-04 00:00:00'), Timestamp('2024-06-15 00:00:00'), Timestamp('2024-02-11 00:00:00'), Timestamp('2024-08-31 00:00:00'), Timestamp('2023-02-11 00:00:00'), Timestamp('2022-04-17 00:00:00')]


In [24]:
import pandas as pd

# Baca file submission kamu dan sample submission
sample_submission = pd.read_csv('../comodity-price-prediction-penyisihan-arkavidia-9/Harga Bahan Pangan/train/Bawang Merah.csv')
sample_submission['Date'] = pd.to_datetime(sample_submission['Date'])
df_natural_gas['Date'] = pd.to_datetime(df_natural_gas['Date'])

# Bandingkan id yang ada di sample submission tapi tidak ada di submission
missing_ids = set(sample_submission['Date']) - set(df_natural_gas['Date'])

# Tampilkan jumlah dan contoh id yang hilang
print(f"Jumlah id yang hilang: {len(missing_ids)}")
print("Contoh id yang hilang:", list(missing_ids)[:10])

Jumlah id yang hilang: 278
Contoh id yang hilang: [Timestamp('2022-12-31 00:00:00'), Timestamp('2022-01-30 00:00:00'), Timestamp('2024-02-17 00:00:00'), Timestamp('2024-04-21 00:00:00'), Timestamp('2024-02-04 00:00:00'), Timestamp('2024-06-15 00:00:00'), Timestamp('2024-02-11 00:00:00'), Timestamp('2024-08-31 00:00:00'), Timestamp('2023-02-11 00:00:00'), Timestamp('2022-04-17 00:00:00')]


In [39]:
for key, df_sorted in df_dict_sorted.items():
    filename = f"{key}_clean.csv"  # Tambahkan "_clean" di akhir nama file
    df_sorted.to_csv(filename, index=False)  # Simpan sebagai CSV tanpa index
    print(f"File {filename} berhasil disimpan.")

File Newcastle Coal Futures Historical Data.csv_clean.csv berhasil disimpan.
File US Sugar 11 Futures Historical Data.csv_clean.csv berhasil disimpan.
File Crude Oil WTI Futures Historical Data.csv_clean.csv berhasil disimpan.
File US Wheat Futures Historical Data.csv_clean.csv berhasil disimpan.
File Palm Oil Futures Historical Data.csv_clean.csv berhasil disimpan.
File Natural Gas Futures Historical Data.csv_clean.csv berhasil disimpan.
